# 🌿 Diagnostic SVP — Score de Verdure et Proximité

Ce notebook permet de :
- Vérifier l'intégrité des données à chaque couche
- Visualiser la distribution des scores
- Débugger les cas limites
- Explorer les données gold avant intégration API/front

**Prérequis** : avoir lancé `python run_pipeline.py --indicateur SVP`

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from shapely import from_wkt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Imports OK')

## 1. Vérification couche Bronze

In [ ]:
def check_bronze(name, path):
    p = Path(path)
    if not p.exists():
        print(f'❌ {name} : fichier manquant → {p}')
        return None
    df = pd.read_parquet(p)
    print(f'✅ {name}')
    print(f'   Lignes    : {len(df):,}')
    print(f'   Colonnes  : {list(df.columns)}')
    if 'geometry_wkt' in df.columns:
        n_null = df['geometry_wkt'].isna().sum()
        print(f'   Géom. nulles : {n_null}')
    print()
    return df

ev_b  = check_bronze('Espaces verts (bronze)', 'data/bronze/bronze_SVP/espaces_verts_raw.parquet')
arb_b = check_bronze('Arbres (bronze)',         'data/bronze/bronze_SVP/arbres_raw.parquet')
com_b = check_bronze('Commerces (bronze)',       'data/bronze/bronze_SVP/commerces_alim_raw.parquet')

## 2. Vérification couche Silver

In [ ]:
def load_silver_gdf(path):
    df = pd.read_parquet(path)
    geom = df['geometry_wkt'].apply(lambda w: from_wkt(w) if pd.notna(w) else None)
    return gpd.GeoDataFrame(df.drop(columns=['geometry_wkt']), geometry=geom, crs='EPSG:4326')

# Espaces verts
ev_s = load_silver_gdf('data/silver/silver_SVP/espaces_verts_clean.parquet')
print(f'Espaces verts silver : {len(ev_s):,} | surface médiane : {ev_s["surface_m2"].median():,.0f} m²')
print(ev_s.head(3))
print()

In [ ]:
# Arbres
arb_s = load_silver_gdf('data/silver/silver_SVP/arbres_clean.parquet')
print(f'Arbres silver : {len(arb_s):,}')
if 'typeemplacement' in arb_s.columns:
    print(arb_s['typeemplacement'].value_counts())
print(arb_s.head(3))

In [ ]:
# Commerces
com_s = load_silver_gdf('data/silver/silver_SVP/commerces_alim_clean.parquet')
print(f'Commerces silver : {len(com_s):,}')
print('Répartition catégories :')
print(com_s['categorie'].value_counts())
print(f'\nPoids moyen par catégorie :')
print(com_s.groupby('categorie')['poids'].mean())

## 3. Données Gold SVP

In [ ]:
gold = pd.read_parquet('data/gold/gold_SVP/svp_par_rue.parquet')
print(f'Gold SVP : {len(gold):,} rues')
print(f'Colonnes : {list(gold.columns)}')
print()
gold.describe().round(2)

In [ ]:
# Distribution des labels
LABELS = ['Très faible', 'Faible', 'Modéré', 'Bon', 'Excellent']
dist = gold['svp_label'].value_counts().reindex(LABELS).fillna(0).astype(int)
print('Distribution des labels SVP :')
for label, n in dist.items():
    pct = n / len(gold) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<15} {n:>5} rues  {pct:5.1f}%  {bar}')

In [ ]:
# Histogramme de distribution des scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(gold['svp_score'], bins=40, color='#4ade80', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribution SVP global', fontweight='bold')
axes[0].set_xlabel('Score SVP')
axes[0].set_ylabel('Nb rues')
axes[0].axvline(gold['svp_score'].median(), color='#16a34a', linestyle='--', label=f'Médiane: {gold["svp_score"].median():.1f}')
axes[0].legend()

axes[1].hist(gold['score_vert'] * 100, bins=40, color='#22c55e', edgecolor='white', linewidth=0.5)
axes[1].set_title('Distribution score_vert', fontweight='bold')
axes[1].set_xlabel('score_vert × 100')

axes[2].hist(gold['score_acces_alim'] * 100, bins=40, color='#fb923c', edgecolor='white', linewidth=0.5)
axes[2].set_title('Distribution score_acces_alim', fontweight='bold')
axes[2].set_xlabel('score_acces_alim × 100')

plt.tight_layout()
plt.savefig('data/gold/gold_SVP/svp_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Sauvegardé : data/gold/gold_SVP/svp_distribution.png')

## 4. Top / Flop rues

In [ ]:
cols_display = ['nom_voie', 'arrondissement', 'nb_espaces_verts', 'nb_arbres', 'score_alim_brut', 'svp_score', 'svp_label']

print('🌿 Top 10 — rues les plus vertes et accessibles')
display(gold.nlargest(10, 'svp_score')[cols_display].reset_index(drop=True))

print('\n🏭 Flop 10 — rues les plus grises et isolées')
display(gold.nsmallest(10, 'svp_score')[cols_display].reset_index(drop=True))

In [ ]:
# Scores médians par arrondissement
by_arr = (
    gold.groupby('arrondissement')
    .agg(
        nb_rues=('svp_score', 'count'),
        svp_median=('svp_score', 'median'),
        ev_moy=('nb_espaces_verts', 'mean'),
        arb_moy=('nb_arbres', 'mean'),
        alim_moy=('score_alim_brut', 'mean'),
    )
    .round(2)
    .sort_values('svp_median', ascending=False)
)
print('SVP médian par arrondissement :')
display(by_arr)

In [ ]:
# Barplot arrondissements
fig, ax = plt.subplots(figsize=(14, 5))

arr_sorted = by_arr.reset_index().sort_values('svp_median', ascending=True)
colors = cm.RdYlGn(arr_sorted['svp_median'] / 100)

bars = ax.barh(arr_sorted['arrondissement'].astype(str) + 'e',
               arr_sorted['svp_median'], color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, arr_sorted['svp_median']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

ax.set_xlabel('Score SVP médian')
ax.set_title('Score SVP médian par arrondissement parisien', fontweight='bold')
ax.set_xlim(0, 105)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('data/gold/gold_SVP/svp_par_arrondissement.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Cas limites — vérification

In [ ]:
# Rues sans aucun espace vert ni arbre
zero_vert = gold[(gold['nb_espaces_verts'] == 0) & (gold['nb_arbres'] == 0)]
print(f'Rues sans végétation dans 200m : {len(zero_vert):,} ({len(zero_vert)/len(gold)*100:.1f}%)')

# Rues sans aucun commerce alimentaire
zero_alim = gold[gold['score_alim_brut'] == 0]
print(f'Rues sans commerce alim dans 500m : {len(zero_alim):,} ({len(zero_alim)/len(gold)*100:.1f}%)')

# Rues avec les deux à zéro (désert total)
zero_total = gold[(gold['nb_espaces_verts'] == 0) & (gold['nb_arbres'] == 0) & (gold['score_alim_brut'] == 0)]
print(f'Rues désert total (0 vert, 0 alim) : {len(zero_total):,}')
if len(zero_total) > 0:
    print('  Exemples :')
    print(zero_total[['nom_voie', 'arrondissement', 'svp_score']].head(5).to_string(index=False))

In [ ]:
# Validation complète
print('=== VALIDATION GOLD SVP ===')
checks = [
    ('Scores dans [0, 100]',   gold['svp_score'].between(0, 100).all()),
    ('Pas de NaN svp_score',   gold['svp_score'].isna().sum() == 0),
    ('Pas de NaN svp_label',   gold['svp_label'].isna().sum() == 0),
    ('Longitudes Paris',       gold['lon_centre'].between(2.2, 2.5).all()),
    ('Latitudes Paris',        gold['lat_centre'].between(48.8, 48.95).all()),
    ('score_vert dans [0,1]',  gold['score_vert'].between(0, 1.001).all()),
    ('score_alim dans [0,1]',  gold['score_acces_alim'].between(0, 1.001).all()),
    ('Spread suffisant',       (gold['svp_score'].max() - gold['svp_score'].min()) > 50),
]
all_ok = True
for label, ok in checks:
    status = '✅' if ok else '❌'
    print(f'  {status} {label}')
    if not ok:
        all_ok = False

print()
print('✅ Toutes les validations OK' if all_ok else '❌ Des validations ont échoué')

## 6. Carte rapide (matplotlib)

In [ ]:
# Scatter plot géographique coloré par score SVP
fig, ax = plt.subplots(figsize=(10, 10))

scatter = ax.scatter(
    gold['lon_centre'], gold['lat_centre'],
    c=gold['svp_score'],
    cmap='RdYlGn',
    s=8,
    alpha=0.7,
    vmin=0, vmax=100
)

cbar = plt.colorbar(scatter, ax=ax, shrink=0.6)
cbar.set_label('Score SVP (0=gris, 100=vert)', fontsize=11)

ax.set_title('Score SVP par rue — Paris intramuros', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_aspect('equal')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('data/gold/gold_SVP/svp_carte.png', dpi=150, bbox_inches='tight')
plt.show()
print('Carte sauvegardée : data/gold/gold_SVP/svp_carte.png')

## 7. Test de l'API (si le serveur tourne)

In [ ]:
import requests

BASE = 'http://localhost:8000'

try:
    r = requests.get(f'{BASE}/svp/', timeout=3)
    print('✅ API accessible')
    print(r.json())
except Exception as e:
    print(f'❌ API non accessible : {e}')
    print('   Lancer : uvicorn api.main:app --reload --port 8000')

In [ ]:
# Test des endpoints SVP
endpoints = [
    '/svp/stats',
    '/svp/rues?label=Excellent&limit=5',
    '/svp/rues?arrondissement=12&sort_by=svp_score&order=desc&limit=5',
]

for ep in endpoints:
    try:
        r = requests.get(f'{BASE}{ep}', timeout=5)
        data = r.json()
        print(f'✅ {ep}')
        if 'rues' in data:
            print(f'   → {data["count"]} résultats')
        elif 'nb_rues_total' in data:
            print(f'   → {data["nb_rues_total"]} rues | médiane SVP: {data["svp_score_median"]}')
    except Exception as e:
        print(f'❌ {ep} : {e}')